In [2]:
import torch

x = torch.tensor([1.0, 2.0, 3.0])
print(x.shape, x.dtype, x.device)

B, T, D = 2, 8, 16
a = torch.zeros(B, T, D)    # 全零
b = torch.randn(B, T, D)      # 标准正态分布

def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = pick_device()
c = torch.ones(3, 4, device = device,dtype = torch.float32)
print(c.device)

torch.Size([3]) torch.float32 cpu
cuda:0


In [6]:
import numpy as np 
import torch

arr = np.random.randn(4, 8).astype(np.float32) # NumPy 创建浮点默认是 float64,torch 默认 float32

# from_numpy 与 arr共享底层内存,改一方可能影响另一方
x = torch.from_numpy(arr)
arr[0, 0] = 999.0
print(x[0, 0])

# 需独立副本时用torch.tensor 或 clone
y = torch.tensor(arr)
arr[0,0] = 0.0
print(y[0,0],x[0,0])

tensor(999.)
tensor(999.) tensor(0.)


In [11]:
from typing import Tuple

import torch
import torch.nn as nn

def split_heads(x: torch.Tensor, n_heads: int, head_dim: int) -> torch.Tensor:
    b, t ,c = x.shape
    assert c == n_heads*head_dim
    x_view = x.view(b,t,n_heads,head_dim)        # (b,t,c) -> (b,t,nh,dh) -> (b, nh, t, dh)
    return x.view(b,t,n_heads,head_dim).transpose(1,2) 
    

class DummyModel(nn.Module):
    def __init__(self,d_model: int) -> None:
        super().__init__()
        self.proj = nn.Linear(d_model,d_model)
    
    def forward(self,x: torch.Tensor) -> torch.Tensor:
        return self.proj(x)
    
x = torch.Tensor(2,3,30)
n_heads = 5
head_dim = 6
b, t ,c = x.shape
assert c == n_heads*head_dim
x_view = x.view(b,t,n_heads,head_dim)        # (b,t,c) -> (b,t,nh,dh) 
x_trans = x_view.transpose(1,2)              # (b, t, nh, dh)  -> (b,nh,t,dh)
print(x.shape,x.stride())
print(x_view.shape,x_view.stride())
print(x_trans.shape,x_trans.stride())
    

torch.Size([2, 3, 30]) (90, 30, 1)
torch.Size([2, 3, 5, 6]) (90, 30, 6, 1)
torch.Size([2, 5, 3, 6]) (90, 6, 30, 1)


In [12]:
import functools
import time
from typing import Any,Callable,TypeVar

import torch

F = TypeVar("F",bound=Callable[..., Any])

def timeit(fn: F) -> F:
    """包装函数,打印耗时"""

    @functools.wraps(fn)
    def wrapper(*args: Any,**kwargs: Any) -> Any:
        t0 = time.perf_counter()
        out = fn(*args,**kwargs)
        t1 = time.perf_counter()
        print(f"{fn.__name__}: {(t1 - t0) *1000:.2f} ms")
        return out
 
    return wrapper   # type: ignore[return-value]

@torch.no_grad()
def eval_forward(model: torch.nn.Module,x: torch.Tensor) -> torch.Tensor:
    """推理阶段关闭梯度"""
    model.eval()
    return model(x)

In [4]:
import torch 

# 从最后一行对齐,逐维度[相等或其一为1] 可广播

a = torch.randn(32, 1, 128)
b = torch.randn(128)
c = a + b


logits = torch.randn(4, 10, 50257)
bisa = torch.randn(50257).view(1, 1, -1)
out = logits+bisa
print(out.shape)

torch.Size([4, 10, 50257])


In [2]:
import torch

w = torch.randn(10, 1, requires_grad=True)
x = torch.randn(1, 10)
y = (x@w).sum()
print(w.grad_fn)     # grad_fn 只挂在非叶子上,AccmuluateGrad是指向叶子的累加节点
print(y.grad_fn)     # grad_fn是一个指向该张量是如何被计算出来的C++函数节点的指针
y.backward()
print(w.grad.shape) # type: ignore
print(w.grad)

# 梯度清零,否则会累计
w.grad.zero_() # type: ignore

None
torch.Size([10, 1])
tensor([[ 1.6611],
        [ 0.0125],
        [-0.3593],
        [ 1.2878],
        [-0.3483],
        [-0.8841],
        [-0.6709],
        [-0.4799],
        [-0.4444],
        [-1.3641]])


tensor([[0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.]])

In [3]:
import torch
import torch.nn as nn

class TinyLinear(nn.Module):
    def __init__(self,in_features: int,out_features: int):
        super().__init__()
        # 小随机初始化,避免一开始就饱和
        self.weight = nn.Parameter(torch.randn(out_features,in_features)* 0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x @ self.weight.T                  # x@w.T

m = TinyLinear(4, 2)
print("参数个数:", sum(p.numel() for p in m.parameters()))

参数个数: 8


In [4]:
import torch
from einops import rearrange,repeat

B, T, HD = 2, 64, 128
H, D = 8, 16
assert H*D == HD
x = torch.randn(B, T, HD)

# 命名维度,论文里常见的(B, T ,H ,D) 拆分
x_heads = rearrange(x,"b t (h d) -> b h t d",h=H,t=T)
print(x_heads.shape)

y = torch.randn(3, 1)
y_rep = repeat(y, "a b -> a (repeat b)",repeat=4)
print(y_rep.shape)

torch.Size([2, 8, 64, 16])
torch.Size([3, 4])


In [5]:
def param_memory_gb(num_params: int, bytes_per_param: int = 4) -> float:
    """仅权重占用,不含优化器状态与激活"""
    return num_params*bytes_per_param / (1024**3)

def matmul_flops_2mnk(m: int,n: int, k: int) -> int:
    """矩阵乘C = A@B,形状 (m,k)@(k,n); 按2*M*N*K计算FLOPs的一种常见口径"""
    return 2*m*n*k

n = 1_000_000_000
print(f"1B 参数 FP32权重约 {param_memory_gb(n, 4):.2f}GB")
print(f"1B 参数 BF16权重约 {param_memory_gb(n, 2):.2f}GB")

M, N, K = 4096, 4096, 4096
print("4096^3 matmul FLOPs (2MNK): ",matmul_flops_2mnk(M, N, K))

1B 参数 FP32权重约 3.73GB
1B 参数 BF16权重约 1.86GB
4096^3 matmul FLOPs (2MNK):  137438953472


In [9]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class MyLinear(nn.Module):
    def __init__(self,in_f: int, out_f: int):
        super().__init__()
        self.weigt = nn.Parameter(torch.randn(out_f,in_f,device = device)*0.01)
        self.bias = nn.Parameter(torch.zeros(out_f,device=device))
    
    def forward(self,x: torch.Tensor) -> torch.Tensor:
        return x @ self.weigt.T + self.bias

torch.manual_seed(0)
model = MyLinear(5, 3).to(device)
opt = torch.optim.SGD(model.parameters(), lr = 0.1)

x = torch.randn(4, 5, device=device)
target = torch.randn(4, 3, device=device)

for step in range(3):
    opt.zero_grad(set_to_none=True)
    pred = model(x)
    loss = (pred-target).pow(2).mean()
    loss.backward()
    opt.step()
    print(step,loss)

0 tensor(0.8845, device='cuda:0', grad_fn=<MeanBackward0>)
1 tensor(0.7624, device='cuda:0', grad_fn=<MeanBackward0>)
2 tensor(0.6800, device='cuda:0', grad_fn=<MeanBackward0>)


In [10]:
import torch

def describe(t: torch.Tensor, name: str) -> None:
    print(name, "shape", tuple(t.shape), "stride", t.stride(), "contig", t.is_contiguous())

x = torch.randn(2, 3, 4)
describe(x, "x")
y = x.transpose(1, 2)
describe(y, "transpose")
describe(y.contiguous(), "contiguous")

x shape (2, 3, 4) stride (12, 4, 1) contig True
transpose shape (2, 4, 3) stride (12, 1, 4) contig False
contiguous shape (2, 4, 3) stride (12, 3, 1) contig True
